In [0]:
import pandas as pd

base_path = "/Workspace/Users/baodi.wilkinson.external@atradius.com/SAP_BO_Analytics/Rivendell_Analysis"

df_brokers = pd.read_csv(f"{base_path}/KPIs_Brokers_050326.csv")
df_portfolio = pd.read_csv(f"{base_path}/KPIS_Portfolio_050326.csv")
df_sales = pd.read_csv(f"{base_path}/KPIs_Sales_cluster_7_050326.csv")

# Fix column name typo: Data_Profider_Refresh_Time -> Data_Provider_Refresh_Time
df_brokers = df_brokers.rename(columns={'Data_Profider_Refresh_Time': 'Data_Provider_Refresh_Time'})
df_portfolio = df_portfolio.rename(columns={'Data_Profider_Refresh_Time': 'Data_Provider_Refresh_Time'})
df_sales = df_sales.rename(columns={'Data_Profider_Refresh_Time': 'Data_Provider_Refresh_Time'})

print(f"Brokers: {df_brokers.shape}")
print(f"Portfolio: {df_portfolio.shape}")
print(f"Sales: {df_sales.shape}")

display(df_brokers.head())
display(df_portfolio.head())
display(df_sales.head())

In [0]:
import numpy as np

# Compare schemas across the 3 DataFrames
frames = {'Brokers': df_brokers, 'Portfolio': df_portfolio, 'Sales': df_sales}

# 1. Column consistency check
print("=" * 60)
print("1. COLUMN CONSISTENCY CHECK")
print("=" * 60)
cols_brokers = set(df_brokers.columns)
cols_portfolio = set(df_portfolio.columns)
cols_sales = set(df_sales.columns)

if cols_brokers == cols_portfolio == cols_sales:
    print("V All 3 DataFrames have identical column names")
else:
    print("X Column differences found:")
    all_cols = cols_brokers | cols_portfolio | cols_sales
    for col in sorted(all_cols):
        present = []
        if col in cols_brokers: present.append('Brokers')
        if col in cols_portfolio: present.append('Portfolio')
        if col in cols_sales: present.append('Sales')
        if len(present) < 3:
            print(f"  {col}: only in {present}")

print(f"\nColumn count: Brokers={len(df_brokers.columns)}, Portfolio={len(df_portfolio.columns)}, Sales={len(df_sales.columns)}")

# 2. Data type comparison
print("\n" + "=" * 60)
print("2. DATA TYPE COMPARISON")
print("=" * 60)
dtype_comparison = pd.DataFrame({
    'Brokers': df_brokers.dtypes.astype(str),
    'Portfolio': df_portfolio.dtypes.astype(str),
    'Sales': df_sales.dtypes.astype(str)
})
dtype_mismatches = dtype_comparison[dtype_comparison.apply(lambda row: len(set(row)) > 1, axis=1)]
if dtype_mismatches.empty:
    print("V All columns have consistent types across DataFrames")
else:
    print(f"X {len(dtype_mismatches)} columns have type mismatches:")
    print(dtype_mismatches.to_string())

# 3. Null/NaN analysis
print("\n" + "=" * 60)
print("3. NULL/NaN ANALYSIS")
print("=" * 60)
null_summary = pd.DataFrame({
    'Brokers_nulls': df_brokers.isnull().sum(),
    'Brokers_pct': (df_brokers.isnull().sum() / len(df_brokers) * 100).round(1),
    'Portfolio_nulls': df_portfolio.isnull().sum(),
    'Portfolio_pct': (df_portfolio.isnull().sum() / len(df_portfolio) * 100).round(1),
    'Sales_nulls': df_sales.isnull().sum(),
    'Sales_pct': (df_sales.isnull().sum() / len(df_sales) * 100).round(1),
})
# Show only columns with at least some nulls
null_cols = null_summary[(null_summary['Brokers_pct'] > 0) | (null_summary['Portfolio_pct'] > 0) | (null_summary['Sales_pct'] > 0)]
print(f"Columns with nulls: {len(null_cols)} of {len(df_brokers.columns)}")
print(null_cols.to_string())

# 4. Duplicate check
print("\n" + "=" * 60)
print("4. DUPLICATE ROWS CHECK")
print("=" * 60)
for name, df in frames.items():
    dups = df.duplicated().sum()
    print(f"  {name}: {dups} duplicate rows ({dups/len(df)*100:.1f}%)")

# 5. Key column value ranges
print("\n" + "=" * 60)
print("5. KEY COLUMN VALUE OVERVIEW")
print("=" * 60)
for name, df in frames.items():
    print(f"\n--- {name} ({len(df)} rows) ---")
    print(f"  documentId: {df['documentId'].nunique()} unique, range [{df['documentId'].min()} - {df['documentId'].max()}]")
    print(f"  dataType_basic: {df['dataType_basic'].unique().tolist()}")
    print(f"  qualification_basic: {df['qualification_basic'].unique().tolist()}")
    print(f"  Document_name: {df['Document_name'].nunique()} unique")

In [0]:
# Add Rivendell category column and concatenate all 3 DataFrames
df_brokers['Rivendell_category'] = 'Brokers'
df_portfolio['Rivendell_category'] = 'Portfolio'
df_sales['Rivendell_category'] = 'Sales'

df_combined = pd.concat([df_brokers, df_portfolio, df_sales], ignore_index=True)

print(f"Combined DataFrame: {df_combined.shape}")
print(f"Category counts:\n{df_combined['Rivendell_category'].value_counts().to_string()}")
# display(df_combined.head())

In [0]:
from datetime import datetime

# Add ingestion timestamp
df_combined['ingestion_ts'] = datetime.now()

# Write to UC table
target_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Rivendel_KPI_Mar26"

df_spark = spark.createDataFrame(df_combined)
df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

print(f"Data written to {target_table}")
print(f"Row count: {spark.table(target_table).count()}")

### Analysis of Rivendel Reports with Cluster and Users Audit 

In [0]:
# Get distinct report Document_Ids from Rivendel_KPI_Mar26
# Join to audit table to find who uses them
# Enrich with user profiles and reporting hierarchy

df_reports = spark.sql("""
  WITH rivendel_reports AS (
    SELECT DISTINCT Document_Id, Document_name, Rivendell_category
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Rivendel_KPI_Mar26
  ),
  
  report_users AS (
    SELECT 
      r.Document_Id,
      r.Document_name,
      r.Rivendell_category,
      a.UserName,
      a.event_type,
      COUNT(*) AS access_count,
      MAX(a.Access_TS) AS last_access
    FROM rivendel_reports r
    INNER JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit a
      ON r.Document_Id = a.WEBI_Doc_ID
    GROUP BY r.Document_Id, r.Document_name, r.Rivendell_category, a.UserName, a.event_type
  ),
  
  user_details AS (
    SELECT 
      ru.*,
      up.full_name,
      up.email,
      up.BU,
      up.level,
      up.line_manager,
      up.AD_status
    FROM report_users ru
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles up
      ON ru.UserName = up.user_ID
  )
  
  SELECT 
    ud.Document_Id,
    ud.Document_name,
    ud.Rivendell_category,
    ud.UserName,
    ud.full_name,
    ud.email,
    ud.BU,
    ud.level,
    ud.line_manager,
    ud.AD_status,
    ud.event_type,
    ud.access_count,
    ud.last_access,
    h.L0_name AS top_manager,
    h.L0_businessUnit AS top_manager_BU,
    h.L0_uid AS top_manager_id,
    h.L1_name AS L1_manager,
    h.L1_businessUnit AS L1_manager_BU,
    h.L2_name AS L2_manager,
    h.L2_businessUnit AS L2_manager_BU,
    h.L3_name AS L3_manager,
    mgr.BU AS line_manager_BU
  FROM user_details ud
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_hierarchies h
    ON ud.UserName = COALESCE(h.L7_uid, h.L6_uid, h.L5_uid, h.L4_uid, h.L3_uid, h.L2_uid, h.L1_uid)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles mgr
    ON array_join(array_sort(split(UPPER(TRIM(ud.line_manager)), ' ')), ' ')
     = array_join(array_sort(split(UPPER(TRIM(mgr.full_name)), ' ')), ' ')
  ORDER BY ud.Rivendell_category, ud.Document_name, ud.access_count DESC
""")

print(f"Total rows: {df_reports.count()}")
print(f"Distinct reports: {df_reports.select('Document_Id').distinct().count()}")
print(f"Distinct users: {df_reports.select('UserName').distinct().count()}")
display(df_reports)

# Convert to pandas for aggregations
df_chart = df_reports.toPandas()

# Users with no line manager found
print("\n" + "=" * 60)
print("0. USERS WITH NO LINE MANAGER")
print("=" * 60)
df_no_mgr = df_chart[df_chart['line_manager'].isna()][['UserName', 'full_name', 'BU']].drop_duplicates()
df_no_mgr = df_no_mgr.sort_values('BU')
df_no_mgr.columns = ['UserName', 'full_name', 'BusinessUnit']
print(f"Total users with no line manager: {len(df_no_mgr)}")
display(spark.createDataFrame(df_no_mgr))

# Total unique users by manager levels (fill nulls with 'EMPTY')
print("\n" + "=" * 60)
print("1. UNIQUE USERS BY LINE MANAGER")
print("=" * 60)
df_by_line_mgr = df_chart.copy()
df_by_line_mgr['line_manager'] = df_by_line_mgr['line_manager'].fillna('EMPTY')
df_by_line_mgr['line_manager_BU'] = df_by_line_mgr['line_manager_BU'].fillna('EMPTY')
df_by_line_mgr = df_by_line_mgr.groupby(['line_manager', 'line_manager_BU'])['UserName'].nunique().reset_index()
df_by_line_mgr.columns = ['line_manager', 'BusinessUnit', 'unique_users']
df_by_line_mgr = df_by_line_mgr.sort_values('unique_users', ascending=False)
print(f"Total distinct line managers: {len(df_by_line_mgr)}")
display(spark.createDataFrame(df_by_line_mgr))

print("\n" + "=" * 60)
print("2. UNIQUE USERS BY L2 MANAGER")
print("=" * 60)
df_by_l2 = df_chart.copy()
df_by_l2['L2_manager'] = df_by_l2['L2_manager'].fillna('EMPTY')
df_by_l2['L2_manager_BU'] = df_by_l2['L2_manager_BU'].fillna('EMPTY')
df_by_l2 = df_by_l2.groupby(['L2_manager', 'L2_manager_BU'])['UserName'].nunique().reset_index()
df_by_l2.columns = ['L2_manager', 'BusinessUnit', 'unique_users']
df_by_l2 = df_by_l2.sort_values('unique_users', ascending=False)
print(f"Total distinct L2 managers: {len(df_by_l2)}")
display(spark.createDataFrame(df_by_l2))

# Bar chart for L2 Manager (excluding EMPTY)
import matplotlib.pyplot as plt
import numpy as np

df_l2_chart = df_by_l2[df_by_l2['L2_manager'] != 'EMPTY'].sort_values('unique_users', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(df_l2_chart['L2_manager'] + ' (' + df_l2_chart['BusinessUnit'] + ')', 
               df_l2_chart['unique_users'], color='#2196F3')
ax.set_xlabel('Unique Users')
ax.set_title('Unique Users by L2 Manager')

# Add value labels
for bar in bars:
    ax.annotate(f'{int(bar.get_width())}', 
                xy=(bar.get_width(), bar.get_y() + bar.get_height() / 2),
                ha='left', va='center', fontsize=10, xytext=(5, 0), textcoords='offset points')

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("3. UNIQUE USERS BY L1 MANAGER")
print("=" * 60)
df_by_l1 = df_chart.copy()
df_by_l1['L1_manager'] = df_by_l1['L1_manager'].fillna('EMPTY')
df_by_l1['L1_manager_BU'] = df_by_l1['L1_manager_BU'].fillna('EMPTY')
df_by_l1 = df_by_l1.groupby(['L1_manager', 'L1_manager_BU'])['UserName'].nunique().reset_index()
df_by_l1.columns = ['L1_manager', 'BusinessUnit', 'unique_users']
df_by_l1 = df_by_l1.sort_values('unique_users', ascending=False)
print(f"Total distinct L1 managers: {len(df_by_l1)}")
display(spark.createDataFrame(df_by_l1))

print("\n" + "=" * 60)
print("4. UNIQUE USERS BY TOP MANAGER (L0)")
print("=" * 60)
df_by_l0 = df_chart.copy()
df_by_l0['top_manager'] = df_by_l0['top_manager'].fillna('EMPTY')
df_by_l0['top_manager_BU'] = df_by_l0['top_manager_BU'].fillna('EMPTY')
df_by_l0 = df_by_l0.groupby(['top_manager', 'top_manager_BU'])['UserName'].nunique().reset_index()
df_by_l0.columns = ['top_manager', 'BusinessUnit', 'unique_users']
df_by_l0 = df_by_l0.sort_values('unique_users', ascending=False)
print(f"Total distinct top managers: {len(df_by_l0)}")
display(spark.createDataFrame(df_by_l0))

# Bar chart: unique users and unique reports by Rivendell_category
categories = sorted(df_chart['Rivendell_category'].dropna().unique())

unique_users = [df_chart[df_chart['Rivendell_category'] == cat]['UserName'].nunique() for cat in categories]
unique_reports = [df_chart[df_chart['Rivendell_category'] == cat]['Document_Id'].nunique() for cat in categories]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, unique_users, width, label='Unique Users', color='#2196F3')
bars2 = ax.bar(x + width/2, unique_reports, width, label='Unique Reports', color='#FF9800')

ax.set_xlabel('Rivendell Category')
ax.set_ylabel('Count')
ax.set_title('Unique Users and Reports by Rivendell Category')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()

# Add value labels on bars
for bar in bars1:
    ax.annotate(f'{bar.get_height()}', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha='center', va='bottom', fontsize=11)
for bar in bars2:
    ax.annotate(f'{bar.get_height()}', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt

# Get Rivendel document IDs and their cluster assignments
df_cluster = spark.sql("""
  SELECT 
    CAST(r.Document_Id AS STRING) AS Document_Id,
    CAST(l.cluster AS INT) AS cluster
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Rivendel_KPI_Mar26 r
  INNER JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_ucflagged_domain l
    ON CAST(r.Document_Id AS STRING) = l.Document_Id
""").toPandas()

# Count distinct Document_Ids per cluster
df_counts = df_cluster.groupby('cluster')['Document_Id'].nunique().reset_index()
df_counts.columns = ['cluster', 'document_count']
df_counts = df_counts.sort_values('cluster')

print(f"Total distinct Rivendel documents found in clusters: {df_cluster['Document_Id'].nunique()}")
print(f"Clusters represented: {sorted(df_counts['cluster'].tolist())}")

# Bar chart
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(df_counts['cluster'].astype(str), df_counts['document_count'], color='#2196F3')

ax.set_xlabel('Cluster')
ax.set_ylabel('Distinct Document Count')
ax.set_title('Rivendel Document ID Counts by Cluster')

# Add value labels
for bar in bars:
    ax.annotate(f'{int(bar.get_height())}', 
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()